RAG with Multi Data Source

In [ ]:
!pip install arxiv

In [ ]:
## Import Wikipedia tools

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [ ]:
## Create Wikipedia tool

api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=200
)

wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

In [ ]:
wiki.name

In [ ]:
## Load website and create FAISS vector database

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()

documents = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
).split_documents(docs)

vectordb = FAISS.from_documents(
    documents,
    OpenAIEmbeddings()
)

retriever = vectordb.as_retriever()

retriever

In [ ]:
## Create retriever tool

from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search for information about LangSmith. For any questions about LangSmith, you must use this tool!"
)

In [ ]:
retriever_tool.name

In [ ]:
## Create Arxiv tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=200
)

arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)

arxiv.name

In [ ]:
tools = [wiki, arxiv, retriever_tool]

In [ ]:
## Configure OpenAI API key and LLM

from dotenv import load_dotenv
import os

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-3.5-turbo-0125",
    temperature=0
)

In [ ]:
## Load agent prompt

from langchain import hub

# Get the prompt to use
prompt = hub.pull("hwchase17/openai-functions-agent")

prompt.messages

In [ ]:
## Create the agent

from langchain.agents import create_openai_tools_agent

agent = create_openai_tools_agent(
    llm,
    tools,
    prompt
)

In [ ]:
## Create Agent Executor

from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

agent_executor

In [ ]:
agent_executor.invoke({
    "input": "Tell me about Langsmith"
})

In [ ]:
agent_executor.invoke({
    "input": "What's the paper 1605.08386 about?"
})